In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_pnl_with_strikes_and_breakevens(legs, width=20, points=500):
    """
    Plot PnL at expiration for multiple option legs, mark PnL at each strike,
    and mark breakeven points where PnL crosses zero.

    Parameters:
    legs : list of dicts
        Each dict should have:
        - 'type': 'call' or 'put'
        - 'strike': strike price (float)
        - 'premium': premium paid/received per contract (float)
        - 'qty': number of contracts (positive for long, negative for short)
    width : float
        How far below/above the min/max strike to extend the price grid.
    points : int
        Number of points in the underlying price grid.
    """
    # Sort strikes
    strikes = sorted([leg['strike'] for leg in legs])

    # Underlying price grid
    S_min = strikes[0] - width
    S_max = strikes[-1] + width
    S = np.linspace(S_min, S_max, points)

    # Compute PnL curve
    pnl = np.zeros_like(S)
    for leg in legs:
        if leg['type'].lower() == 'call':
            payoff = np.maximum(S - leg['strike'], 0)
        else:  # put
            payoff = np.maximum(leg['strike'] - S, 0)
        pnl += leg['qty'] * (payoff - leg['premium'])

    # Compute PnL at strikes
    pnl_at_strikes = []
    for K in strikes:
        val = sum(
            leg['qty'] * ((max(K - leg['strike'], 0) if leg['type']=='call' else max(leg['strike'] - K, 0))
                          - leg['premium'])
            for leg in legs
        )
        pnl_at_strikes.append(val)

    # Find breakeven (zero-crossing) points via linear interpolation
    be_points = []
    for i in range(len(S)-1):
        if pnl[i] == 0:
            be_points.append(S[i])
        elif pnl[i]*pnl[i+1] < 0:
            # linear interpolate root between i and i+X
            x0, x1 = S[i], S[i+1]
            y0, y1 = pnl[i], pnl[i+1]
            root = x0 - y0*(x1-x0)/(y1-y0)
            be_points.append(root)

    # Plot PnL curve
    plt.figure(figsize=(8,5))
    plt.plot(S, pnl, linewidth=2, label='PnL Curve')

    # Mark PnL at strikes
    plt.scatter(strikes, pnl_at_strikes, color='C1', zorder=5, label='PnL @ Strikes')
    for x, y in zip(strikes, pnl_at_strikes):
        plt.text(x, y, f'{y:.2f}', fontsize=8, va='bottom', ha='center')

    # Mark breakevens
    plt.scatter(be_points, [0]*len(be_points), color='red', marker='x', s=75, zorder=6, label='Breakevens')
    for x in be_points:
        plt.text(x, 0, f'{x:.2f}', fontsize=8, va='bottom', ha='left', color='red')

    plt.axhline(0, color='black', linewidth=0.5)
    plt.xlabel('Underlying Price at Expiration')
    plt.ylabel('Profit / Loss')
    plt.title('PnL with Strike Values and Breakevens')
    plt.grid(True, linestyle='--', linewidth=0.5)
    plt.xlim(left=0)
    plt.legend()
    plt.show()

    # Print out values
    print("PnL at strike prices:")
    for K, val in zip(strikes, pnl_at_strikes):
        print(f"  Strike = {K}, PnL = {val:.2f}")
    print("\nBreakeven (PnL=0) underlying prices:")
    for x in be_points:
        print(f"  Price = {x:.2f}")

if __name__ == "__main__":
    # Example legs: replace with your own

    call_option_chain = {
        19: 0.77,
        18.5: 0.9,
        18: 1.08,
        17.5: 1.33,
        17: 1.64,
        16.5: 2.01,
        16: 2.45,
        15.5: 2.91
    }

    put_option_chain = {
        19: 1.37,
        18.5: 1.01,
        18: 0.69,
        17.5: 0.45,
        17: 0.26,
        16.5: 0.13,
        16: 0.07,
        15.5: 0.04
    }

    legs = [
        {'type': 'put', 'strike': 40, 'premium': 3.0, 'qty': 1},
        {'type': 'put', 'strike': 45, 'premium': 5.0, 'qty': -2},
        {'type': 'put', 'strike': 50, 'premium': 8.0, 'qty': 1},
    ]
    plot_pnl_with_strikes_and_breakevens(legs)
